# Exercise 2: Exploratory Data Analysis
## Patronizing and Condescending Language (PCL) Detection

This notebook performs exploratory data analysis on the *Don't Patronize Me!* dataset (SemEval-2022 Task 4). We investigate two key characteristics of the data that directly inform our modelling decisions:

1. **Class Distribution & Community Keyword Analysis** -- understanding imbalance and community-level variation.
2. **Text Length Distribution Analysis** -- understanding how paragraph length differs between PCL and non-PCL samples, and what this means for tokeniser configuration.

In [ ]:
import os
import ast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.15)
plt.rcParams["figure.dpi"] = 150

# Paths (relative to src/)
DATA_PATH = os.path.join("..", "data", "dontpatronizeme_pcl.tsv")
TRAIN_SPLIT = os.path.join("..", "data", "practice-splits", "train_semeval_parids-labels.csv")
DEV_SPLIT = os.path.join("..", "data", "practice-splits", "dev_semeval_parids-labels.csv")
FIG_DIR = os.path.join("..", "report", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

## Data Loading

The TSV file has a 4-line disclaimer header and no column names. We load it directly and assign column names. The split files map `par_id` values to annotator-level labels; we use these to partition the data into train and dev sets.

In [ ]:
# Load the main dataset (skip 4-line disclaimer, no header row)
df = pd.read_csv(
    DATA_PATH,
    sep="\t",
    skiprows=4,
    header=None,
    names=["par_id", "art_id", "keyword", "country", "text", "label"],
)

# Binary label: 0-1 -> No PCL (0), 2-4 -> PCL (1)
df["pcl"] = (df["label"] >= 2).astype(int)

# Load train/dev splits
train_ids = pd.read_csv(TRAIN_SPLIT)["par_id"].values
dev_ids = pd.read_csv(DEV_SPLIT)["par_id"].values

df["split"] = "unused"
df.loc[df["par_id"].isin(train_ids), "split"] = "train"
df.loc[df["par_id"].isin(dev_ids), "split"] = "dev"

print(f"Total paragraphs: {len(df):,}")
print(f"Train: {(df['split'] == 'train').sum():,}  |  Dev: {(df['split'] == 'dev').sum():,}")
print(f"\nLabel distribution (original 0-4 scale):")
print(df["label"].value_counts().sort_index())
print(f"\nBinary distribution:")
print(df["pcl"].value_counts().rename({0: "No PCL", 1: "PCL"}))
df.head()

---
## Technique 1: Class Distribution & Community Keyword Analysis

We first visualise the overall class imbalance, then drill down to see how the prevalence of PCL varies across the 10 community keyword categories.

In [ ]:
# ── Overall class distribution ──────────────────────────────────────────────
counts = df["pcl"].value_counts().sort_index()
labels_map = {0: "No PCL", 1: "PCL"}
ratio = counts[0] / counts[1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Bar chart
palette = ["#4c72b0", "#dd8452"]
bars = axes[0].bar(
    [labels_map[k] for k in counts.index],
    counts.values,
    color=palette,
    edgecolor="black",
    linewidth=0.7,
)
for bar, count in zip(bars, counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 80,
        f"{count:,}",
        ha="center",
        va="bottom",
        fontweight="bold",
        fontsize=12,
    )
axes[0].set_ylabel("Number of paragraphs")
axes[0].set_title(f"Binary Class Distribution (ratio {ratio:.1f}:1)")
axes[0].set_ylim(0, counts.max() * 1.15)

# Pie chart
axes[1].pie(
    counts.values,
    labels=[labels_map[k] for k in counts.index],
    autopct="%1.1f%%",
    colors=palette,
    startangle=90,
    textprops={"fontsize": 12},
    wedgeprops={"edgecolor": "black", "linewidth": 0.7},
)
axes[1].set_title("Class Proportions")

fig.suptitle("Overall Class Distribution: PCL vs No PCL", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "class_distribution.png"), bbox_inches="tight")
plt.show()

print(f"Imbalance ratio (No PCL : PCL) = {ratio:.1f} : 1")

In [ ]:
# ── PCL rate by community keyword ──────────────────────────────────────────
community_stats = (
    df.groupby("keyword")
    .agg(total=("pcl", "count"), pcl_count=("pcl", "sum"))
    .assign(pcl_rate=lambda d: d["pcl_count"] / d["total"] * 100)
    .sort_values("pcl_rate", ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5.5))
bar_colors = sns.color_palette("YlOrRd", n_colors=len(community_stats))
bars = ax.barh(
    community_stats.index,
    community_stats["pcl_rate"],
    color=bar_colors,
    edgecolor="black",
    linewidth=0.6,
)

# Annotate bars
for bar, (_, row) in zip(bars, community_stats.iterrows()):
    ax.text(
        bar.get_width() + 0.4,
        bar.get_y() + bar.get_height() / 2,
        f"{row['pcl_rate']:.1f}%  ({int(row['pcl_count'])}/{int(row['total'])})",
        va="center",
        fontsize=10,
    )

ax.set_xlabel("PCL Rate (%)")
ax.set_title("PCL Prevalence by Community Keyword", fontsize=14, fontweight="bold")
ax.set_xlim(0, community_stats["pcl_rate"].max() * 1.35)
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "community_pcl_rates.png"), bbox_inches="tight")
plt.show()

# Summary table
print("\nCommunity keyword PCL statistics:")
print(community_stats.to_string())

### Analysis -- Technique 1

**Findings:**
- The dataset exhibits a severe class imbalance of approximately **10:1** (No PCL vs PCL). Only around 9-10% of paragraphs are labelled as containing patronizing or condescending language.
- PCL prevalence varies substantially across community keywords. Some communities (e.g., *disabled*, *vulnerable*, *in-need*) tend to attract higher rates of patronizing language, while others (e.g., *immigrant*, *migrant*) have lower PCL rates. This reflects how certain framings of vulnerability are more prone to condescension in media discourse.

### Impact on Classification Approach

1. **Focal loss over downsampling.** The ~10:1 imbalance means naive training will produce a classifier that overwhelmingly predicts the majority class. Simple downsampling would discard roughly 90% of the negative examples, wasting valuable signal about what *non-patronizing* language looks like. Focal loss (Lin et al., 2017) dynamically down-weights easy/majority-class examples during training, allowing the model to learn from the full dataset while focusing gradient updates on the harder, minority-class samples.

2. **Community keyword as an input feature.** The pronounced variation in PCL rate across communities suggests that a model could benefit from knowing which community is being discussed. Prepending the keyword (e.g., `"disabled [SEP] paragraph text"`) as an additional input signal gives the transformer explicit community context, potentially improving discrimination in ambiguous cases.

---
## Technique 2: Text Length Distribution Analysis

We compare the word-count distributions of PCL and non-PCL paragraphs to understand whether text length is a discriminating feature and to inform our choice of `max_sequence_length` for transformer tokenisation.

In [ ]:
# Compute word counts
df["word_count"] = df["text"].astype(str).str.split().str.len()

pcl_lengths = df.loc[df["pcl"] == 1, "word_count"]
no_pcl_lengths = df.loc[df["pcl"] == 0, "word_count"]

# ── Summary statistics ──────────────────────────────────────────────────────
stats = (
    df.groupby("pcl")["word_count"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .rename(index={0: "No PCL", 1: "PCL"})
)
# Add 95th percentile
stats["p95"] = df.groupby("pcl")["word_count"].quantile(0.95).values
stats = stats.round(1)
print("Text length statistics (word count):")
print(stats.to_string())

In [ ]:
# ── Overlapping histograms + violin plot ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: overlapping histograms
bins = np.arange(0, df["word_count"].quantile(0.99) + 10, 10)
axes[0].hist(
    no_pcl_lengths, bins=bins, alpha=0.6, label="No PCL", color="#4c72b0",
    edgecolor="black", linewidth=0.4, density=True,
)
axes[0].hist(
    pcl_lengths, bins=bins, alpha=0.6, label="PCL", color="#dd8452",
    edgecolor="black", linewidth=0.4, density=True,
)

# Mark medians
axes[0].axvline(no_pcl_lengths.median(), color="#4c72b0", ls="--", lw=1.5, label=f"No PCL median ({no_pcl_lengths.median():.0f})")
axes[0].axvline(pcl_lengths.median(), color="#dd8452", ls="--", lw=1.5, label=f"PCL median ({pcl_lengths.median():.0f})")

axes[0].set_xlabel("Word count")
axes[0].set_ylabel("Density")
axes[0].set_title("Normalised Histograms")
axes[0].legend(fontsize=9)

# Right: violin plot
plot_df = df[["pcl", "word_count"]].copy()
plot_df["class"] = plot_df["pcl"].map({0: "No PCL", 1: "PCL"})
sns.violinplot(
    data=plot_df, x="class", y="word_count", palette=["#4c72b0", "#dd8452"],
    inner="quartile", ax=axes[1], cut=0,
)
axes[1].set_xlabel("")
axes[1].set_ylabel("Word count")
axes[1].set_title("Violin Plot")

fig.suptitle("Text Length Distributions: PCL vs No PCL", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "text_length_dist.png"), bbox_inches="tight")
plt.show()

# ── 95th percentile summary ────────────────────────────────────────────────
overall_p95 = df["word_count"].quantile(0.95)
print(f"\nOverall 95th percentile word count: {overall_p95:.0f} words")
print(f"Approximate token count at 95th percentile (x1.3 sub-word expansion): ~{overall_p95 * 1.3:.0f} tokens")

### Analysis -- Technique 2

**Findings:**
- PCL paragraphs tend to be **longer** than non-PCL paragraphs. The median word count for PCL texts is noticeably higher than for non-PCL texts, and the PCL distribution has a heavier right tail.
- This makes intuitive sense: patronizing language often involves elaborate justifications, unsolicited advice, or excessive framing that inflates paragraph length. Short factual snippets are less likely to contain condescension.
- However, word count alone is not a reliable classifier -- the distributions overlap substantially.

### Impact on Classification Approach

The 95th percentile word count (across both classes) guides the `max_sequence_length` hyperparameter for transformer tokenisation. Assuming a sub-word expansion factor of roughly 1.3x (typical for RoBERTa on English text), we can estimate the token-level 95th percentile. Setting `max_sequence_length` to capture this percentile ensures we preserve almost all textual content without excessive padding, balancing information retention against GPU memory and training speed. A value of **256 tokens** is a reasonable starting point, with the option to increase to 512 if ablation studies show benefit from capturing the longest paragraphs.

---
## Summary

| Technique | Key Finding | Modelling Implication |
|-----------|------------|----------------------|
| Class Distribution & Community Analysis | ~10:1 imbalance; PCL rate varies significantly by community keyword | Use focal loss (not downsampling); prepend community keyword to input |
| Text Length Distribution | PCL paragraphs are systematically longer; 95th percentile guides sequence length | Set `max_sequence_length` to capture 95th percentile (~256 tokens) |